In [1]:
from alpaca.trading.client import TradingClient

In [2]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)
alpaca_api_key = os.getenv("ALPACA_API_KEY")
alpaca_secret_key = os.getenv("ALPACA_SECRET_KEY")
trading_client= TradingClient(alpaca_api_key, alpaca_secret_key, paper=True)
clock=trading_client.get_clock()

In [3]:
clock

{   'is_open': False,
    'next_close': datetime.datetime(2026, 6, 22, 16, 0, tzinfo=TzInfo(-14400)),
    'next_open': datetime.datetime(2026, 6, 22, 9, 30, tzinfo=TzInfo(-14400)),
    'timestamp': datetime.datetime(2026, 6, 21, 9, 43, 46, 919076, tzinfo=TzInfo(-14400))}

In [4]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockLatestTradeRequest

client = StockHistoricalDataClient(alpaca_api_key, alpaca_secret_key)
request=StockLatestTradeRequest(symbol_or_symbols="AAPL")
trade=client.get_stock_latest_trade(request)

In [8]:
trade["AAPL"].price

297.86

<module 'mcp_params' from '/home/efa/Traders/mcp_params.py'>

In [21]:
from agents.mcp import MCPServerStdio
import importlib
import mcp_params
importlib.reload(mcp_params)
all_params = mcp_params.researcher_mcp_server_params("ed")+mcp_params.trader_mcp_server_params
count = 0
for each_params in all_params:
    async with MCPServerStdio(params=each_params, client_session_timeout_seconds=60) as server:
        mcp_tools = await server.list_tools()
        count += len(mcp_tools)
print(f"We have {len(all_params)} MCP servers, and {count} tools")

We have 6 MCP servers, and 38 tools


In [19]:
from agents.mcp import MCPServerStdio
import os
alpaca_params = {
      "command": "uvx",
      "args": ["alpaca-mcp-server"],
      "env": {
        "ALPACA_API_KEY": os.getenv("ALPACA_API_KEY"),
        "ALPACA_SECRET_KEY": os.getenv("ALPACA_SECRET_KEY"),
          "ALPACA_TOOLSETS": "stock-data,crypto-data,options-data,index-data,news"
      }
    }

params={"command": "python", "args": [ "tavily_server.py"]}
async with MCPServerStdio(params=alpaca_params, client_session_timeout_seconds=120) as server:
    tools=await server.list_tools()

In [20]:
len(tools)

25

# Trying a trader agent


In [4]:
from accounts import Account
from IPython.display import display, Markdown
import importlib
import accounts_client
importlib.reload(accounts_client)

ed_initial_strategy = "You are a day trader that buys and sells shares based on news and market conditions."
Account.get("Ed").reset(ed_initial_strategy)

display(Markdown(await accounts_client.read_accounts_resource("Ed")))
display(Markdown(await accounts_client.read_strategy_resource("Ed")))

{"name": "ed", "balance": 10000.0, "strategy": "You are a day trader that buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-06-21 20:58:44", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}

You are a day trader that buys and sells shares based on news and market conditions.

In [ ]:
agent_name = "Ed"

# Using MCP Servers to read resources
account_details = await accounts_client.read_accounts_resource(agent_name)
strategy = await accounts_client.read_strategy_resource(agent_name)

instructions = f"""
You are a trader that manages a portfolio of shares. Your name is {agent_name} and your account is under your name, {agent_name}.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is: {strategy}

Your current holdings and balance is:
{account_details}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
When you buy shares, you need to consdier the current price and you balance to make sure you have enough money to buy the shares. When you sell shares, you need to consider the current price and your holdings to make sure you have enough shares to sell.
You have tools to save memory of companies, research and thinking so far.
Please make use of these tools to manage your portfolio. Carry out trades as you see fit; do not wait for instructions or ask for confirmation.
"""

prompt = """
Use your tools to make decisions about your portfolio.
Investigate the news and the market, make your decision, make the trades, and respond with a summary of your actions.
"""

In [31]:
print(strategy)

You are a day trader that aggressively buys and sells shares based on news and market conditions.


In [3]:
print(instructions)


You are a trader that manages a portfolio of shares. Your name is Ed and your account is under your name, Ed.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is: You are a day trader that aggressively buys and sells shares based on news and market conditions.

Your current holdings and balance is:
{"name": "ed", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-06-21 20:56:54", 10000.0], ["2026-06-21 20:56:59", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking so far.
Pleas

In [35]:
researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in mcp_params.researcher_mcp_server_params("Ed")]
trader_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in mcp_params.trader_mcp_server_params]
mcp_servers = trader_mcp_servers + researcher_mcp_servers

In [39]:
from traders import get_researcher_tool,get_model
model=get_model("groq:openai/gpt-oss-120b")

In [40]:
model

In [43]:
from agents import Agent,Runner,trace
from traders import get_researcher_tool,get_model
for server in mcp_servers:
    await server.connect()

researcher_tool = await get_researcher_tool(researcher_mcp_servers,"gpt-4o-mini")
trader = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool],
    mcp_servers=trader_mcp_servers,
    model="gpt-4o-mini",
)
with trace(agent_name):
    result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

CancelledError: 